In [1]:
import json
import sys
from rich import print as rp
from pathlib import Path
import re
from datetime import datetime
import pandas as pd
import openpyxl

nb_dir = Path.cwd()
project_root = nb_dir.parent.parent
sys.path.insert(0, str(project_root))

In [2]:
remaining_cids_file = Path(project_root / "scripts/notebooks/remaining_cids.json")
with open(remaining_cids_file, "r") as f:
   remaining_list = json.load(f)

insel_missing_people = [i for i in remaining_list if "insel" in i]

# with open("insel_missing_people.json", "w") as f:
#     json.dump(insel_missing_people, f, ensure_ascii=False, indent=2)


In [3]:
insel_details_file = Path(project_root / "scripts/notebooks/insel_details_orig.json")
with open(insel_details_file, "r") as f:
   insel_details_orig = json.load(f)

rp(insel_details_orig[:1])

# insel_with_nrs = []
# insel_no_nrs = []
check_people = []
insel_details = []
insel_found = []

for book in insel_details_orig:
    book_id = book["book_id"]
    original_entry = book["original_entry"]
    pid = book["person_id"]
    cid = book["composite_id"]
    find_ib = re.search(r"IB[\s.–-]*(\d+)", original_entry)
    if find_ib:
        ib_nr = find_ib.group(1)
    else:
        ib_nr = "x"

    if cid in insel_missing_people and pid:
        insel_found.append({**book, "ib_nr": ib_nr})
    if cid in insel_missing_people and not pid:
        check_people.append({**book, "ib_nr": ib_nr})

    insel_details.append({
        **book,
        "ib_nr": ib_nr
    })

# rp(len(insel_details))
# rp(f"checks needed: {len(check_people)}")
# rp(insel_found)

# with open("insel_found.json", "w") as f:
#     json.dump(insel_found, f, ensure_ascii=False, indent=2)

# with open("insel_missing_people.json", "w") as f:
#     json.dump(check_people, f, ensure_ascii=False, indent=2)

[
    {
        'book_id': 4549,
        'composite_id': 'insel_226_10_27',
        'title': 'ES WAR EINMAL',
        'subtitle': None,
        'copies': 1,
        'amount': 10,
        'original_entry': 'ES WAR EINMAL IB 360\nEin Bilderbuch von Ludwig Richter. Mit 48 Abbildungen. Leipzig 
Insel 1936. 71 S. Sehr gut erhalten.',
        'person_id': 5524,
        'unified_id': 'richter_ludwig',
        'family_name': 'Richter',
        'given_names': 'Ludwig',
        'display_name': 'Richter, Ludwig'
    }
]

In [4]:
insel_build_csv = []
for book in insel_details:
    book_id = book["book_id"]
    pid = book["person_id"]
    cid = book["composite_id"]
    title = book["title"]
    subtitle = book["subtitle"]
    copies = book["copies"]
    amount = book["amount"]
    family_name = book["family_name"]
    given_names = book["given_names"]
    display_name = book["display_name"]
    ib_nr = book["ib_nr"]

    if not subtitle:
        titel = title or ""
    else:
        titel = title + ". " + subtitle

    if not pid and not display_name:
        autor = ""
    elif family_name and given_names:
        autor = family_name + ", " + given_names
    elif family_name and not given_names:
        autor = family_name
    elif display_name and not family_name and not given_names:
        autor = display_name
    else:
        autor = display_name or ""

    if not copies or copies == 1:
        exemplare = ""
    else:
        exemplare = copies

    if not amount:
        preis = ""
    else:
        preis = amount

    insel_build_csv.append({
        "cid": cid,
        "ib_nr": ib_nr,
        "titel": titel,
        "autor": autor,
        "preis": preis,
        "exemplare": exemplare,
    })

# rp(insel_build_csv[:30])

In [7]:
df = pd.DataFrame(insel_build_csv)

df = df.rename(columns={
    "ib_nr": "IB-Nr.",
    "titel": "Titel",
    "autor": "Autor",
    "preis": "Preis",
    "exemplare": "Exemplare",
    })
df = df[["IB-Nr.", "Titel", "Autor", "Preis", "Exemplare"]]

df["_sort"] = pd.to_numeric(df["IB-Nr."], errors="coerce").astype("Int64")

df = df.sort_values("_sort", na_position="last")
df = df.drop(columns="_sort")
df.head(10)

# df.to_excel("insel_details.xlsx", index=False)
# df.to_csv("insel_details.csv", index=False, encoding="utf-8")
df.to_csv("insel_details.txt", index=False, encoding="utf-8")